# Feature Attribution using Raking - v1.0

In [1]:
import sys

!{sys.executable} -m pip install xgboost
!{sys.executable} -m pip install catboost

In [2]:
import time
import numpy as np
import pandas as pd
import zipfile as zf
import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.sparse.linalg import eigs
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

# Data preprocessing methods

In [3]:
# return one df without nan's on num_cols
# numerical features: fill nan's with median/mean/mode
def pre_proc_fillna_num_fts(df,num_cols,num_type='mean'):
    df_train= df.copy()

    if(num_type=='median'):
        for col in num_cols:
            ft_median= df_train[col].median()
            df_train[col]= df_train[col].fillna(ft_median)
    elif(num_type=='mode'):
        for col in num_cols:
            ft_mode= df_train[col].value_counts().index[0]
            df_train[col]= df_train[col].fillna(ft_mode)
    else:
        for col in num_cols:
            ft_mean= df_train[col].mean()
            df_train[col]= df_train[col].fillna(ft_mean)

    return df_train

In [4]:
# return one df without nan's on cat_cols
# categorical features: fill nan's with mode/mean/median
def pre_proc_fillna_cat_fts(df,cat_cols,cat_type='mode'):
    df_train= df.copy()
    
    if(cat_type!='mode' and type(df_train[cat_cols[0]].value_counts().index[0])!=type('str')):
        if(cat_type=='mean'):
            for col in cat_cols:
                ft_mean= df_train[col].mean()
                df_train[col]= df_train[col].fillna(ft_mean)
        elif(cat_type=='median'):
            for col in cat_cols:
                ft_median= df_train[col].median()
                df_train[col]= df_train[col].fillna(ft_median)
    else:
        for col in cat_cols:
            ft_mode= df_train[col].value_counts().index[0]
            df_train[col]= df_train[col].fillna(ft_mode)

    return df_train

In [5]:
# n_cols refers only to df's columns with numerical values
def normalize_selected_cols(df, n_cols):
    result= df.copy()
    
    for col in n_cols:
        max_value= df[col].max()
        min_value= df[col].min()
        result[col]= (df[col]- min_value)/ (max_value - min_value)
        
    return result

# Feature Attribution using Raking methods

In [6]:
# return a DataFrame with replace values (mean/median/mode/none to categorical and numeric) to fill train cols. df is post-processed (cat_cols encoded)
def replace_values(df,num_cols,num_type='mean',cat_type='none'):
    cat_values= None
    num_values= None
    
    if (cat_type=='mean'):
        cat_values= df.mean(axis=0).to_frame().T
    elif (cat_type=='median'):
        cat_values= df.median(axis=0).to_frame().T
    elif (cat_type=='mode'):
        cat_values= df.mode(axis=0)
    
    if (num_type=='mode'):
        num_values= df.mode(axis=0)
    elif (num_type=='median'):
        num_values= df.median(axis=0).to_frame().T
    elif (num_type=='mean'):
        num_values= df.mean(axis=0).to_frame().T

    if(cat_type!='none'):
        cat_values[num_cols]= num_values[num_cols]
        return cat_values
    
    return num_values

In [7]:
# train the ML model and return its mean accuracy after n_train runs
def train_model_get_acc_mean(model, x_trn, x_tst, y_trn, y_tst, n_train):
    trainings= []

    for i in range(n_train):

        model.fit(x_trn, y_trn)
        acc= sklearn.metrics.accuracy_score(y_tst, model.predict(x_tst))

        trainings.append(acc)

    return np.mean(trainings)

In [8]:
# re-training is needed because machine learning models typically assume that the train and the test data comes from a similar distribution 
# (Hooker et al., 2018)
# here we return p(x|i) and p(x|ij)
def remove_and_retrain_v1(model, replace_ft_vals, x_trn, x_tst, y_trn, y_tst, n_train):
    acc_no_i= []
    acc_no_ij= []

    n_fts= len(x_trn.columns)

    start= time.time()

    for i in range(n_fts):
        acc_row= []

        # replace the i-th ft with its respective mode/mean to "remove" it. train the ML model and get the mean accuracy
        train_copy_no_i= x_trn.copy()
        train_copy_no_i.loc[:,train_copy_no_i.columns[i]]= replace_ft_vals.iloc[0,i]

        acc_no_i.append(train_model_get_acc_mean(model, train_copy_no_i, x_tst, y_trn, y_tst, n_train))

        for j in range(n_fts):

            if (i!= j):
                # replace the j-th ft with its respective mode/mean to "remove" it. here, we "remove" the i-th and the j-th ft
                # train the ML model and get the mean accuracy
                train_copy_no_ij= train_copy_no_i.copy()
                train_copy_no_ij.loc[:,train_copy_no_ij.columns[j]]= replace_ft_vals.iloc[0,j]

                acc_row.append(train_model_get_acc_mean(model, train_copy_no_ij, x_tst, y_trn, y_tst, n_train))
            else:
                acc_row.append(0)

        acc_no_ij.append(acc_row)

    end= time.time()
    #print("--- %s seconds ---" % np.round((end- start), 2))

    return acc_no_i, acc_no_ij

In [9]:
# here we return | p(x|ij) - p(x|i) |
def get_p_matrix_v1(n_fts, acc_no_i, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))

    for i in range(n_fts):
        pi= acc_no_i[i]
        
        for j in range(n_fts):
            if (i!= j):
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= abs(pij- pi)
                
    return p_matrix

In [10]:
# here we return | p(x|ij) - p(x|j) |
def get_p_matrix_v2(n_fts, acc_no_j, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))

    for i in range(n_fts):
        for j in range(n_fts):
            if (i!= j):
                pj= acc_no_j[j]
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= abs(pij- pj)
                
    return p_matrix

In [11]:
# here we return (| p(x|ij) - p(x|j) | + | p(x|i) - p(x) |) / 2
def get_p_matrix_v3(n_fts, acc_all, acc_no_i, acc_no_j, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))
    p= acc_all

    for i in range(n_fts):
        pi= acc_no_i[i]
        
        for j in range(n_fts):
            if (i!= j):
                pj= acc_no_j[j]
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= (abs(pij- pj)+ abs(pi- p))/ 2
                
    return p_matrix

In [12]:
# here we return (| p(x|ij) - p(x|i) | + | p(x|j) - p(x) |) / 2
def get_p_matrix_v4(n_fts, acc_all, acc_no_i, acc_no_j, acc_no_ij):
    p_matrix= np.zeros((n_fts, n_fts))
    p= acc_all

    for i in range(n_fts):
        pi= acc_no_i[i]
        
        for j in range(n_fts):
            if (i!= j):
                pj= acc_no_j[j]
                pij= acc_no_ij[i][j]
                
                p_matrix[i][j]= (abs(pij- pi)+ abs(pj- p))/ 2
                
    return p_matrix

In [13]:
# the stationary distribution is the fraction of time that the system spends in each state as the number of samples approaches infinity
# it looks like there's not a built-in method to find the stationary distribution

# converts a matrix to a row stochastic matrix - a real square matrix, with each row summing to 1
def to_row_stochastic_matrix(M):
    result= M
    
    for row in result:
        n= sum(row)
        if n> 0:
            row[:]= [f/sum(row) for f in row]
    
    return result

In [14]:
# the stationary distribution - analytical solution
# return 1D array
def stationary_dist_v1(stochastic_matrix):
    
    size_A= stochastic_matrix.shape[1]
    ones= [1]* size_A

    A= np.append(np.transpose(stochastic_matrix)- np.identity(size_A),[ones],axis=0)

    v= np.zeros(size_A+ 1)
    v[size_A]= 1
    v= np.transpose(v)

    stationary= np.linalg.solve(np.transpose(A).dot(A), np.transpose(A).dot(v))

    return stationary

In [15]:
# the stationary distribution - another analytical solution
# return 2D array
def stationary_dist_v2(stochastic_matrix):
    # we have to transpose so that Markov transitions correspond to right multiplying by a column vector
    eigval, eigvec= eigs(stochastic_matrix.T, k=1, which='LM')
    stationary= eigvec/ eigvec.sum()

    # eigs finds complex eigenvalues and eigenvectors, so you'll want the real part.
    stationary= stationary.real

    return stationary

In [16]:
# return a sorted DataFrame with Features and Importances - OHE compacted, that is, dataset's original features
def ft_importance_df(importances, ft_names, replace_list):
    fti= pd.Series(importances, index=ft_names).sort_values(ascending=False).to_frame().reset_index()
    fti= fti.rename(columns= {'index':'Feature',0:'Importance'}, inplace=False)
    fti['Feature'].replace(replace_list, inplace=True)
    fti= fti.groupby(['Feature']).sum().sort_values('Importance', ascending=False).reset_index()
    
    return fti

# Data loading and preprocessing

In [17]:
!kaggle competitions download -c titanic

/bin/bash: kaggle: command not found


In [18]:
ds= zf.ZipFile('datasets/titanic.zip')

train_data= pd.read_csv(ds.open('train.csv'))
test_data= pd.read_csv(ds.open('test.csv'))

train_data.shape, test_data.shape

((891, 12), (418, 11))

In [19]:
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [20]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [21]:
X_all= pd.concat([train_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']],
                   test_data[['PassengerId','Pclass','Sex','Age','SibSp','Parch','Fare','Embarked']]]).set_index('PassengerId')

y_train= train_data[['PassengerId','Survived']].set_index('PassengerId')['Survived']

X_all

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,
1,3,male,22.0,1,0,7.2500,S
2,1,female,38.0,1,0,71.2833,C
3,3,female,26.0,0,0,7.9250,S
4,1,female,35.0,1,0,53.1000,S
5,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...
1305,3,male,NaN,0,0,8.0500,S
1306,1,female,39.0,0,0,108.9000,C
1307,3,male,38.5,0,0,7.2500,S


In [22]:
numeric_columns= ['Age','SibSp','Parch','Fare']
categor_columns= list(filter(lambda x:x not in numeric_columns,X_all.columns))

X_train= X_all.iloc[:len(train_data),:].copy()
X_test= X_all.iloc[len(train_data):].copy()

In [23]:
# in this case we'll only use X_train df. X_test is not labeled

In [24]:
X_train= pre_proc_fillna_num_fts(X_train,numeric_columns,num_type='median')

X_train= pre_proc_fillna_cat_fts(X_train,categor_columns,cat_type='mode')

X_train

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,
1,3,male,22.0,1,0,7.2500,S
2,1,female,38.0,1,0,71.2833,C
3,3,female,26.0,0,0,7.9250,S
4,1,female,35.0,1,0,53.1000,S
5,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...
887,2,male,27.0,0,0,13.0000,S
888,1,female,19.0,0,0,30.0000,S
889,3,female,28.0,1,2,23.4500,S


In [25]:
# one-hot encoding the qualitative features
X_train_ohe= pd.get_dummies(X_train,columns=categor_columns)

X_train_ohe.head()

,Age,SibSp,Parch,Fare,Pclass_1,Pclass_2,Pclass_3,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
PassengerId,,,,,,,,,,,,
1,22.0,1,0,7.2500,0,0,1,0,1,0,0,1
2,38.0,1,0,71.2833,1,0,0,1,0,1,0,0
3,26.0,0,0,7.9250,0,0,1,1,0,0,0,1
4,35.0,1,0,53.1000,1,0,0,1,0,0,0,1
5,35.0,0,0,8.0500,0,0,1,0,1,0,0,1


In [26]:
y_train.head()

PassengerId
1    0
2    1
3    1
4    1
5    0
Name: Survived, dtype: int64

In [27]:
# normalize the numeric columns of dataframe with each value between 0 and 1
X_train= normalize_selected_cols(X_train, numeric_columns)
X_train_ohe= normalize_selected_cols(X_train_ohe, numeric_columns)

# ML model setup

In [43]:
train, test, labels_train, labels_test= train_test_split(X_train_ohe,y_train,train_size=0.80,random_state=1234)
rf= sklearn.ensemble.RandomForestClassifier(n_estimators=500,n_jobs=2)

#train, test, labels_train, labels_test= train_test_split(X_train_ohe,y_train,train_size=0.80,random_state=1234)
#xgb_model= xgb.XGBRFClassifier(n_estimators=500,eval_metric='logloss',use_label_encoder=False)

#train, test, labels_train, labels_test= train_test_split(X_train,y_train,train_size=0.80,random_state=1234)
#cat_fts= [train.columns.to_list().index(col) for col in categor_columns]
#ctb_model= CatBoostClassifier(cat_features=cat_fts,silent=True)

# --- Run local modeling ---

In [44]:
X_to_split= X_train_ohe.copy()
Y_to_split= y_train.copy()

In [45]:
target_index= 1

target_instance= X_to_split.loc[X_to_split.index== target_index]
target_label= Y_to_split.loc[Y_to_split.index== target_index]

X_no_targt= X_to_split.drop(target_instance.index)
Y_no_targt= Y_to_split.drop(index= target_index)

In [46]:
Y_no_targt.head()

PassengerId
2    1
3    1
4    1
5    0
6    0
Name: Survived, dtype: int64

In [47]:
from sklearn.neighbors import NearestNeighbors

# find the test_size nearest neighbors from a target_X instance
# return train, test, labels_train, labels_test sets based on knn, with target_X and target_Y into testX and testY
def knn_train_test_split(df_X, df_Y, target_X, target_Y, test_size= 0.2):

    neighbors= math.floor(0.2 * df_X.shape[0])

    knn= NearestNeighbors(n_neighbors= neighbors)
    knn.fit(df_X)

    knn_index= knn.kneighbors(target_X, return_distance=False)
    
    trainX= df_X.drop(df_X.index[knn_index[0]])
    trainY= df_Y.drop(df_Y.index[knn_index[0]])

    testX= df_X.loc[df_X.index[knn_index[0]]]
    testY= df_Y.loc[df_Y.index[knn_index[0]]]

    testX= pd.concat([target_X, testX])
    testY= pd.concat([target_Y, testY])
    
    return trainX, testX, trainY, testY

In [55]:
import math
from sklearn.model_selection import KFold, StratifiedKFold

# df_X and df_Y doesn't contain target_X and target_Y
def kfoldnn_remove_and_retrain(model, df_X, df_Y, target_X, target_Y, numeric_columns, num_type='mean', cat_type='median', test_size=0.2):

    n_folds= math.floor(np.sqrt(df_X.shape[1]))

    if (n_folds< 5):
        n_folds= 5

    skf= StratifiedKFold(n_splits= n_folds, random_state=1234, shuffle=True)

    acc_all= []
    acc_no_i= np.zeros(df_X.shape[1])
    acc_no_ij= np.zeros((df_X.shape[1],df_X.shape[1]))

    repeat_train= 1

    for train_index, test_index in skf.split(df_X, df_Y):

        X_fold= df_X.iloc[train_index]
        Y_fold= df_Y.iloc[train_index]

        # Distance matrix, knn, entire fold?

        train, test, labels_train, labels_test= knn_train_test_split(X_fold, Y_fold, target_X, target_Y, test_size= test_size)

        acc_all.append(train_model_get_acc_mean(model, train, test, labels_train, labels_test, repeat_train))

        replace_ft= replace_values(X_fold, numeric_columns, num_type=num_type, cat_type=cat_type)

        aux_acc_no_i, aux_acc_no_ij= remove_and_retrain_v1(model, replace_ft, train, test, labels_train, labels_test, repeat_train)

        acc_no_i += aux_acc_no_i
        acc_no_ij += aux_acc_no_ij


    acc_no_i /= n_folds
    acc_no_ij /= n_folds

    mean_acc_all= np.mean(acc_all)
    
    return mean_acc_all, acc_no_i, acc_no_ij

In [56]:
mean_acc_all, acc_no_i, acc_no_ij= kfoldnn_remove_and_retrain(rf, X_no_targt, Y_no_targt, target_instance, 
                                                              target_label, numeric_columns, num_type='mean', 
                                                              cat_type='median', test_size= 0.2)

In [57]:
print(mean_acc_all)
print("-----------------------------")
#print(acc_no_i)
print(np.around(acc_no_i, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij, decimals=3))

0.8713286713286713
-----------------------------
[0.878 0.862 0.856 0.452 0.857 0.871 0.871 0.857 0.873 0.855 0.874 0.873]
-----------------------------
[[0.    0.876 0.836 0.87  0.877 0.874 0.874 0.874 0.877 0.877 0.874 0.876]
 [0.883 0.    0.871 0.436 0.877 0.863 0.874 0.876 0.877 0.873 0.877 0.876]
 [0.834 0.874 0.    0.46  0.857 0.873 0.873 0.856 0.857 0.871 0.883 0.873]
 [0.87  0.441 0.441 0.    0.357 0.512 0.476 0.382 0.473 0.379 0.494 0.519]
 [0.88  0.866 0.859 0.369 0.    0.871 0.881 0.874 0.859 0.862 0.867 0.873]
 [0.874 0.876 0.857 0.471 0.86  0.    0.871 0.871 0.873 0.86  0.863 0.871]
 [0.874 0.874 0.873 0.51  0.88  0.877 0.    0.874 0.873 0.866 0.871 0.87 ]
 [0.877 0.862 0.859 0.459 0.881 0.856 0.874 0.    0.662 0.87  0.877 0.873]
 [0.877 0.86  0.864 0.457 0.864 0.859 0.874 0.655 0.    0.871 0.877 0.876]
 [0.88  0.873 0.871 0.373 0.856 0.862 0.866 0.869 0.871 0.    0.871 0.787]
 [0.874 0.866 0.86  0.492 0.863 0.864 0.878 0.883 0.881 0.859 0.    0.874]
 [0.878 0.876 0.866 0.

In [58]:
# get the probability matrix
p_matrix1= get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

#print(p_matrix1)
print(np.around(p_matrix1, decimals=3))
print("-----------------------------")
#print(p_matrix2)
print(np.around(p_matrix2, decimals=3))
print("-----------------------------")
#print(p_matrix3)
print(np.around(p_matrix3, decimals=3))
print("-----------------------------")
#print(p_matrix4)
print(np.around(p_matrix4, decimals=3))

[[0.    0.003 0.042 0.008 0.001 0.004 0.004 0.004 0.001 0.001 0.004 0.003]
 [0.021 0.    0.01  0.425 0.015 0.001 0.013 0.014 0.015 0.011 0.015 0.014]
 [0.022 0.018 0.    0.396 0.001 0.017 0.017 0.    0.001 0.015 0.027 0.017]
 [0.418 0.011 0.011 0.    0.095 0.06  0.024 0.07  0.021 0.073 0.042 0.067]
 [0.022 0.008 0.001 0.488 0.    0.014 0.024 0.017 0.001 0.004 0.01  0.015]
 [0.003 0.004 0.014 0.4   0.011 0.    0.    0.    0.001 0.011 0.008 0.   ]
 [0.003 0.003 0.001 0.361 0.008 0.006 0.    0.003 0.001 0.006 0.    0.001]
 [0.02  0.004 0.001 0.399 0.024 0.001 0.017 0.    0.196 0.013 0.02  0.015]
 [0.004 0.013 0.008 0.415 0.008 0.014 0.001 0.218 0.    0.001 0.004 0.003]
 [0.025 0.018 0.017 0.481 0.001 0.007 0.011 0.014 0.017 0.    0.017 0.067]
 [0.    0.008 0.014 0.382 0.011 0.01  0.004 0.008 0.007 0.015 0.    0.   ]
 [0.006 0.003 0.007 0.392 0.008 0.001 0.    0.011 0.003 0.073 0.001 0.   ]]
-----------------------------
[[0.    0.014 0.02  0.418 0.02  0.003 0.003 0.017 0.004 0.022 0.    0

In [59]:
# finding the stationary distribution
st_matrix1= np.asarray(p_matrix1)
st_matrix2= np.asarray(p_matrix2)
st_matrix3= np.asarray(p_matrix3)
st_matrix4= np.asarray(p_matrix4)

# now convert to right stochastic matrix - a real square matrix, with each row summing to 1
st_matrix1= to_row_stochastic_matrix(st_matrix1)
st_matrix2= to_row_stochastic_matrix(st_matrix2)
st_matrix3= to_row_stochastic_matrix(st_matrix3)
st_matrix4= to_row_stochastic_matrix(st_matrix4)

# get the stationary distribution
stationary_d1= stationary_dist_v1(st_matrix1)
stationary_d2= stationary_dist_v1(st_matrix2)
stationary_d3= stationary_dist_v1(st_matrix3)
stationary_d4= stationary_dist_v1(st_matrix4)

print(stationary_d1)
print("-----------------------------")
print(stationary_d2)
print("-----------------------------")
print(stationary_d3)
print("-----------------------------")
print(stationary_d4)

[0.18133297 0.01914383 0.10866485 0.36259842 0.0482377  0.04142039
 0.02772497 0.05225316 0.02938651 0.04685461 0.03736167 0.04502093]
-----------------------------
[0.02890788 0.04561479 0.04659028 0.15497519 0.06660147 0.0463498
 0.02993716 0.21618155 0.1952339  0.08436362 0.03476624 0.05047811]
-----------------------------
[0.06900872 0.07377442 0.07449794 0.12577662 0.07930447 0.07103539
 0.06863888 0.10560757 0.10197257 0.08495464 0.06881778 0.076611  ]
-----------------------------
[0.14529634 0.03668806 0.03880356 0.35007337 0.05456907 0.05555674
 0.04702091 0.055537   0.05284754 0.04974085 0.05397517 0.05989139]


In [60]:
rp_list= {'Sex_male':'Sex','Sex_female':'Sex','Pclass_1':'Pclass','Pclass_2':'Pclass','Pclass_3':'Pclass',
          'Embarked_S':'Embarked','Embarked_Q':'Embarked','Embarked_C':'Embarked'}

In [75]:
# TARGET INSTANCE TO EXPLAIN ITS FEATURES

X_all.loc[X_all.index== target_index]

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
PassengerId,,,,,,,
1,3,male,22.0,1,0,7.25,S


In [77]:
target_label

PassengerId
1    0
Name: Survived, dtype: int64

In [61]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d1, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Fare          0.362598
Age           0.181333
Parch         0.108665
Sex_female    0.052253
Pclass_1      0.048238
Embarked_C    0.046855
Embarked_S    0.045021
Pclass_2      0.041420
Embarked_Q    0.037362
Sex_male      0.029387
Pclass_3      0.027725
SibSp         0.019144
dtype: float64

In [62]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Fare,0.362598
1,Age,0.181333
2,Embarked,0.129237
3,Pclass,0.117383
4,Parch,0.108665
5,Sex,0.081640
6,SibSp,0.019144


In [63]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d2, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Sex_female    0.216182
Sex_male      0.195234
Fare          0.154975
Embarked_C    0.084364
Pclass_1      0.066601
Embarked_S    0.050478
Parch         0.046590
Pclass_2      0.046350
SibSp         0.045615
Embarked_Q    0.034766
Pclass_3      0.029937
Age           0.028908
dtype: float64

In [64]:
ft_importance_df(stationary_d2,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.411415
1,Embarked,0.169608
2,Fare,0.154975
3,Pclass,0.142888
4,Parch,0.046590
5,SibSp,0.045615
6,Age,0.028908


In [65]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d3, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Fare          0.125777
Sex_female    0.105608
Sex_male      0.101973
Embarked_C    0.084955
Pclass_1      0.079304
Embarked_S    0.076611
Parch         0.074498
SibSp         0.073774
Pclass_2      0.071035
Age           0.069009
Embarked_Q    0.068818
Pclass_3      0.068639
dtype: float64

In [66]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Fare,0.362598
1,Age,0.181333
2,Embarked,0.129237
3,Pclass,0.117383
4,Parch,0.108665
5,Sex,0.081640
6,SibSp,0.019144


In [67]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d4, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Fare          0.350073
Age           0.145296
Embarked_S    0.059891
Pclass_2      0.055557
Sex_female    0.055537
Pclass_1      0.054569
Embarked_Q    0.053975
Sex_male      0.052848
Embarked_C    0.049741
Pclass_3      0.047021
Parch         0.038804
SibSp         0.036688
dtype: float64

In [68]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Fare,0.362598
1,Age,0.181333
2,Embarked,0.129237
3,Pclass,0.117383
4,Parch,0.108665
5,Sex,0.081640
6,SibSp,0.019144


# --- Run global modeling ---

In [34]:
# re-training can result in slightly different models, it is essential to repeat the training process multiple times to ensure that the variance in accuracy is low 
# (Hooker et al., 2018)
repeat_train= 1

num_fts= len(train.columns)

In [35]:
# train the ML model and get the mean accuracy using the entire feature set
acc_all_fts= train_model_get_acc_mean(rf, train, test, labels_train, labels_test, repeat_train)

acc_all_fts

0.8212290502793296

In [36]:
# values to "remove" and retrain
replace_ft= replace_values(train,numeric_columns,num_type='mean',cat_type='median')

replace_ft

,Age,SibSp,Parch,Fare,Pclass_1,Pclass_2,Pclass_3,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,0.362142,0.064782,0.064841,0.064072,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0


In [33]:
# train the ML model and get the mean accuracy removing and retraining columns from the feature set
acc_no_i, acc_no_ij= remove_and_retrain_v1(rf, replace_ft, train, test, labels_train, labels_test, repeat_train)

In [34]:
#print(acc_no_i)
print(np.around(acc_no_i, decimals=3))
print("-----------------------------")
#print(acc_no_ij)
print(np.around(acc_no_ij, decimals=3))

[0.793 0.816 0.821 0.827 0.827 0.821 0.827 0.821 0.821 0.821 0.821 0.821]
-----------------------------
[[0.    0.804 0.832 0.81  0.804 0.793 0.793 0.799 0.804 0.81  0.799 0.793]
 [0.804 0.    0.816 0.799 0.821 0.821 0.821 0.81  0.81  0.816 0.816 0.81 ]
 [0.832 0.821 0.    0.827 0.827 0.827 0.827 0.816 0.821 0.838 0.816 0.821]
 [0.804 0.804 0.821 0.    0.832 0.832 0.827 0.832 0.827 0.827 0.832 0.827]
 [0.799 0.821 0.827 0.838 0.    0.827 0.827 0.821 0.821 0.827 0.827 0.821]
 [0.799 0.821 0.816 0.832 0.827 0.    0.793 0.821 0.821 0.821 0.821 0.816]
 [0.793 0.816 0.827 0.832 0.827 0.793 0.    0.821 0.821 0.821 0.821 0.821]
 [0.804 0.816 0.816 0.827 0.821 0.821 0.816 0.    0.704 0.821 0.821 0.816]
 [0.793 0.81  0.81  0.827 0.821 0.821 0.821 0.682 0.    0.821 0.816 0.81 ]
 [0.804 0.81  0.838 0.821 0.827 0.827 0.821 0.816 0.821 0.    0.816 0.81 ]
 [0.799 0.821 0.827 0.832 0.821 0.821 0.821 0.821 0.821 0.816 0.    0.804]
 [0.799 0.821 0.832 0.832 0.816 0.816 0.821 0.81  0.816 0.821 0.816 0. 

In [35]:
# get the probability matrix
p_matrix1= get_p_matrix_v1(num_fts, acc_no_i, acc_no_ij)
p_matrix2= get_p_matrix_v2(num_fts, acc_no_i, acc_no_ij)
p_matrix3= get_p_matrix_v3(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)
p_matrix4= get_p_matrix_v4(num_fts, acc_all_fts, acc_no_i, acc_no_i, acc_no_ij)

#print(p_matrix1)
print(np.around(p_matrix1, decimals=3))
print("-----------------------------")
#print(p_matrix2)
print(np.around(p_matrix2, decimals=3))
print("-----------------------------")
#print(p_matrix3)
print(np.around(p_matrix3, decimals=3))
print("-----------------------------")
#print(p_matrix4)
print(np.around(p_matrix4, decimals=3))

[[0.    0.011 0.039 0.017 0.011 0.    0.    0.006 0.011 0.017 0.006 0.   ]
 [0.011 0.    0.    0.017 0.006 0.006 0.006 0.006 0.006 0.    0.    0.006]
 [0.011 0.    0.    0.006 0.006 0.006 0.006 0.006 0.    0.017 0.006 0.   ]
 [0.022 0.022 0.006 0.    0.006 0.006 0.    0.006 0.    0.    0.006 0.   ]
 [0.028 0.006 0.    0.011 0.    0.    0.    0.006 0.006 0.    0.    0.006]
 [0.022 0.    0.006 0.011 0.006 0.    0.028 0.    0.    0.    0.    0.006]
 [0.034 0.011 0.    0.006 0.    0.034 0.    0.006 0.006 0.006 0.006 0.006]
 [0.017 0.006 0.006 0.006 0.    0.    0.006 0.    0.117 0.    0.    0.006]
 [0.028 0.011 0.011 0.006 0.    0.    0.    0.14  0.    0.    0.006 0.011]
 [0.017 0.011 0.017 0.    0.006 0.006 0.    0.006 0.    0.    0.006 0.011]
 [0.022 0.    0.006 0.011 0.    0.    0.    0.    0.    0.006 0.    0.017]
 [0.022 0.    0.011 0.011 0.006 0.006 0.    0.011 0.006 0.    0.006 0.   ]]
-----------------------------
[[0.    0.011 0.011 0.017 0.022 0.028 0.034 0.022 0.017 0.011 0.022 0

In [36]:
# finding the stationary distribution
st_matrix1= np.asarray(p_matrix1)
st_matrix2= np.asarray(p_matrix2)
st_matrix3= np.asarray(p_matrix3)
st_matrix4= np.asarray(p_matrix4)

# now convert to right stochastic matrix - a real square matrix, with each row summing to 1
st_matrix1= to_row_stochastic_matrix(st_matrix1)
st_matrix2= to_row_stochastic_matrix(st_matrix2)
st_matrix3= to_row_stochastic_matrix(st_matrix3)
st_matrix4= to_row_stochastic_matrix(st_matrix4)

print(st_matrix1.sum())
print(st_matrix2.sum())
print(st_matrix3.sum())
print(st_matrix4.sum())

12.0
12.0
12.0
12.0


In [37]:
# get the stationary distribution
stationary_d1= stationary_dist_v1(st_matrix1)
stationary_d2= stationary_dist_v1(st_matrix2)
stationary_d3= stationary_dist_v1(st_matrix3)
stationary_d4= stationary_dist_v1(st_matrix4)

print(stationary_d1)
print("-----------------------------")
print(stationary_d2)
print("-----------------------------")
print(stationary_d3)
print("-----------------------------")
print(stationary_d4)

[0.18002355 0.07487006 0.10504361 0.0947893  0.0514202  0.04210641
 0.03612971 0.13734167 0.13324574 0.05965845 0.03834663 0.04702468]
-----------------------------
[0.09688042 0.05350664 0.07679715 0.07196842 0.0418801  0.11985257
 0.10998994 0.14484344 0.13214301 0.03923308 0.04584324 0.067062  ]
-----------------------------
[0.0982633  0.0553632  0.07784017 0.05804254 0.04990515 0.0690063
 0.08915894 0.17159756 0.16051876 0.04758638 0.04997886 0.07273883]
-----------------------------
[0.26717294 0.09732554 0.09698515 0.11084627 0.07909679 0.0311486
 0.05863646 0.07750169 0.07673875 0.05009644 0.02787507 0.0265763 ]


In [38]:
rp_list= {'Sex_male':'Sex','Sex_female':'Sex','Pclass_1':'Pclass','Pclass_2':'Pclass','Pclass_3':'Pclass',
          'Embarked_S':'Embarked','Embarked_Q':'Embarked','Embarked_C':'Embarked'}

In [39]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d1, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Age           0.180024
Sex_female    0.137342
Sex_male      0.133246
Parch         0.105044
Fare          0.094789
SibSp         0.074870
Embarked_C    0.059658
Pclass_1      0.051420
Embarked_S    0.047025
Pclass_2      0.042106
Embarked_Q    0.038347
Pclass_3      0.036130
dtype: float64

In [40]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.270587
1,Age,0.180024
2,Embarked,0.145030
3,Pclass,0.129656
4,Parch,0.105044
5,Fare,0.094789
6,SibSp,0.074870


In [41]:
# feature importance ranking stationary_d1
ft_ranking= pd.Series(stationary_d2, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Sex_female    0.144843
Sex_male      0.132143
Pclass_2      0.119853
Pclass_3      0.109990
Age           0.096880
Parch         0.076797
Fare          0.071968
Embarked_S    0.067062
SibSp         0.053507
Embarked_Q    0.045843
Pclass_1      0.041880
Embarked_C    0.039233
dtype: float64

In [42]:
ft_importance_df(stationary_d2,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.276986
1,Pclass,0.271723
2,Embarked,0.152138
3,Age,0.096880
4,Parch,0.076797
5,Fare,0.071968
6,SibSp,0.053507


In [43]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d3, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Sex_female    0.171598
Sex_male      0.160519
Age           0.098263
Pclass_3      0.089159
Parch         0.077840
Embarked_S    0.072739
Pclass_2      0.069006
Fare          0.058043
SibSp         0.055363
Embarked_Q    0.049979
Pclass_1      0.049905
Embarked_C    0.047586
dtype: float64

In [44]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.270587
1,Age,0.180024
2,Embarked,0.145030
3,Pclass,0.129656
4,Parch,0.105044
5,Fare,0.094789
6,SibSp,0.074870


In [45]:
# feature importance ranking stationary_d2
ft_ranking= pd.Series(stationary_d4, index=X_train_ohe.columns).sort_values(ascending=False)
ft_ranking

Age           0.267173
Fare          0.110846
SibSp         0.097326
Parch         0.096985
Pclass_1      0.079097
Sex_female    0.077502
Sex_male      0.076739
Pclass_3      0.058636
Embarked_C    0.050096
Pclass_2      0.031149
Embarked_Q    0.027875
Embarked_S    0.026576
dtype: float64

In [46]:
ft_importance_df(stationary_d1,X_train_ohe.columns,rp_list)

,Feature,Importance
0,Sex,0.270587
1,Age,0.180024
2,Embarked,0.145030
3,Pclass,0.129656
4,Parch,0.105044
5,Fare,0.094789
6,SibSp,0.074870


In [47]:
eigval, eigvec= eigs(st_matrix3.T, k=1, which='LM')
eigval

array([1.+0.j])

In [48]:
f= open('datasets/FAR_data.txt', 'w')

for i in range(num_fts):
    for j in range(num_fts):
        line= (str(i) + ',' + str(j) + ',' + str(p_matrix1[i][j]) + '\n')
        f.write(line)
        
f.close()